# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [2]:
# Core imports
import os
import asyncio
from typing import Iterator, Optional, Dict, Any

# OpenAI for API calls
import openai
from openai import OpenAI

# Gradio for UI
import gradio as gr

# Additional utilities
import time
import json
from datetime import datetime

# Environment variables (optional - for API key management)
from dotenv import load_dotenv

# Load environment variables if using .env file
load_dotenv()

True

In [3]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging
openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API Key found: {openai_api_key[:8]}...")
else:
    print("Warning: OPENAI_API_KEY not found in environment variables")
    print("Make sure to create a .env file with: OPENAI_API_KEY=your_api_key_here")

# Initialize OpenAI client
client = OpenAI(api_key=openai_api_key)

OpenAI API Key found: sk-proj-...


In [4]:
# Constants
AVAILABLE_MODELS = {
    "GPT-4o": "gpt-4o",
    "GPT-4o Mini": "gpt-4o-mini",
    "GPT-4 Turbo": "gpt-4-turbo-preview",
    "GPT-3.5 Turbo": "gpt-3.5-turbo"
}

DEFAULT_MODEL = "gpt-4o-mini"

SYSTEM_PROMPT = """You are an expert technical educator with deep knowledge across multiple domains including:
- Software engineering and programming
- Data science and machine learning
- System architecture and design
- DevOps and cloud technologies
- Cybersecurity
- Database systems
- Network protocols
- Hardware and embedded systems

Your role is to explain complex technical concepts in a clear, structured, and accessible way. When answering technical questions:

1. Start with a concise summary of the concept
2. Break down complex topics into digestible parts
3. Use analogies and real-world examples when helpful
4. Provide practical context and use cases
5. Include relevant code examples or diagrams when appropriate
6. Explain both the "what" and the "why"
7. Mention common pitfalls or misconceptions
8. Suggest next steps or related topics to explore

Adapt your explanation level based on the complexity of the question, but always aim to be thorough and educational."""

MAX_TOKENS = 2048
TEMPERATURE = 0.7
STREAM_CHUNK_SIZE = 1

In [5]:
# Set up environment
def setup_environment():
    """Initialize and validate the environment setup"""
    
    # Validate OpenAI API key
    if not openai_api_key:
        raise ValueError("OpenAI API key is required. Please set OPENAI_API_KEY in your .env file")
    
    # Test API connection
    try:
        # Make a simple test call to verify the API key works
        test_response = client.models.list()
        print("✅ OpenAI API connection successful")
        print(f"✅ Available models: {len(list(test_response.data))} models found")
    except Exception as e:
        print(f"❌ OpenAI API connection failed: {str(e)}")
        raise
    
    # Validate required models are available
    try:
        models_response = client.models.list()
        available_model_ids = [model.id for model in models_response.data]
        
        missing_models = []
        for display_name, model_id in AVAILABLE_MODELS.items():
            if model_id not in available_model_ids:
                missing_models.append(f"{display_name} ({model_id})")
        
        if missing_models:
            print(f"⚠️  Some models may not be available: {', '.join(missing_models)}")
        else:
            print("✅ All configured models are available")
            
    except Exception as e:
        print(f"⚠️  Could not verify model availability: {str(e)}")
    
    # Set up Gradio theme and configuration
    print("✅ Environment setup complete")
    return True

# Initialize environment on import
try:
    setup_environment()
except Exception as e:
    print(f"⚠️  Environment setup warning: {str(e)}")
    print("The application may not function properly until this is resolved.")


✅ OpenAI API connection successful
✅ Available models: 79 models found
✅ All configured models are available
✅ Environment setup complete


In [6]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [7]:
# Get gpt-4o-mini to answer, with streaming.
def get_technical_explanation(question: str, model: str = DEFAULT_MODEL) -> Iterator[str]:
    """
    Get a technical explanation for a question using OpenAI API with streaming
    
    Args:
        question (str): The technical question to explain
        model (str): The OpenAI model to use (defaults to gpt-4o-mini)
    
    Yields:
        str: Chunks of the explanation as they're generated
    """
    try:
        # Create the streaming chat completion
        stream = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user", 
                    "content": f"Please explain this technical concept or answer this technical question: {question}"
                }
            ],
            max_tokens=MAX_TOKENS,
            temperature=TEMPERATURE,
            stream=True
        )
        
        # Yield each chunk as it arrives
        for chunk in stream:
            if chunk.choices[0].delta.content is not None:
                yield chunk.choices[0].delta.content
                
    except Exception as e:
        error_message = f"Error generating explanation: {str(e)}"
        print(error_message)
        yield error_message

def answer_question_streaming(question: str, model: str = DEFAULT_MODEL) -> Iterator[str]:
    """
    Wrapper function to get streaming technical explanations
    
    Args:
        question (str): The technical question to answer
        model (str): The model to use for generation
        
    Yields:
        str: Progressive chunks of the complete answer
    """
    if not question.strip():
        yield "Please enter a technical question to get an explanation."
        return
    
    print(f"🤔 Generating explanation for: {question[:50]}{'...' if len(question) > 50 else ''}")
    print(f"🤖 Using model: {model}")
    
    full_response = ""
    
    # Stream the response and build the complete answer
    for chunk in get_technical_explanation(question, model):
        full_response += chunk
        yield full_response
    
    print("✅ Explanation generated successfully")

In [10]:
# Gradio Interface
def create_gradio_interface():
    """
    Create and configure the Gradio interface for the technical question explainer
    """
    
    def process_question(question: str, model_choice: str) -> Iterator[str]:
        """
        Process the technical question and stream the response to Gradio
        
        Args:
            question (str): The technical question from the user
            model_choice (str): The selected model display name
            
        Yields:
            str: Streaming response for Gradio display
        """
        # Convert display name to model ID
        model_id = AVAILABLE_MODELS.get(model_choice, DEFAULT_MODEL)
        
        # Stream the response
        yield from answer_question_streaming(question, model_id)
    
    # Create the interface
    with gr.Blocks(
        title="Technical Question Explainer",
        theme=gr.themes.Soft(),
        css="""
        .gradio-container {
            max-width: 900px !important;
            margin: auto !important;
        }
        .main-header {
            text-align: center;
            margin-bottom: 2rem;
        }
        """
    ) as interface:
        
        # Header
        gr.HTML("""
        <div class="main-header">
            <h1>🔬 Technical Question Explainer</h1>
            <p>Get expert-level explanations for complex technical concepts</p>
        </div>
        """)
        
        with gr.Row():
            with gr.Column(scale=3):
                # Input components
                question_input = gr.Textbox(
                    label="Your Technical Question",
                    placeholder="e.g., How does a neural network backpropagate gradients?",
                    lines=3,
                    max_lines=5
                )
                
                model_selector = gr.Dropdown(
                    choices=list(AVAILABLE_MODELS.keys()),
                    value="GPT-4o Mini",
                    label="Select Model",
                    info="Choose the AI model for your explanation"
                )
                
                with gr.Row():
                    submit_btn = gr.Button("🚀 Get Explanation", variant="primary", size="lg")
                    clear_btn = gr.Button("🗑️ Clear", variant="secondary")
            
            with gr.Column(scale=4):
                # Output components
                explanation_output = gr.Textbox(
                    label="Technical Explanation",
                    lines=20,
                    max_lines=25,
                    show_copy_button=True,
                    container=True,
                    interactive=False
                )
        
        # Event handlers
        submit_btn.click(
            fn=process_question,
            inputs=[question_input, model_selector],
            outputs=explanation_output,
            show_progress=True
        )
        
        clear_btn.click(
            fn=lambda: ("", ""),
            outputs=[question_input, explanation_output]
        )
        
        # Allow Enter key to submit
        question_input.submit(
            fn=process_question,
            inputs=[question_input, model_selector],
            outputs=explanation_output,
            show_progress=True
        )
        
        # Examples section
        gr.Examples(
            examples=[
                ["How does TCP/IP work and why is it important?", "GPT-4o Mini"],
                ["Explain the difference between SQL and NoSQL databases", "GPT-4o Mini"],
                ["What is machine learning overfitting and how do you prevent it?", "GPT-4o"],
                ["How do microservices differ from monolithic architecture?", "GPT-4o"],
                ["What are the key principles of secure coding?", "GPT-4 Turbo"]
            ],
            inputs=[question_input, model_selector],
            outputs=explanation_output,
            fn=process_question,
            cache_examples=False
        )
        
        # Footer
        gr.HTML("""
        <div style="text-align: center; margin-top: 2rem; color: #666;">
            <p>💡 Tip: Be specific in your questions for more targeted explanations</p>
        </div>
        """)
    
    return interface

# Launch the application
def main():
    """
    Main function to launch the Gradio application
    """
    print("🚀 Starting Technical Question Explainer...")
    
    # Create and launch the interface
    interface = create_gradio_interface()
    
    # Launch with custom settings
    interface.launch(
        share=True,  # Set to True if you want a public link
      #  server_name="0.0.0.0",  # Allow external access
       # server_port=7860,  # Default Gradio port
       # show_error=True,  # Show detailed error messages
        #quiet=False  # Show startup logs
    )

# Run the application if this script is executed directly
if __name__ == "__main__":
    main()

🚀 Starting Technical Question Explainer...
* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://010d490d635eb2c03e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🤔 Generating explanation for: How do compilers work?
🤖 Using model: gpt-4-turbo-preview
✅ Explanation generated successfully
